# vepyr — VEP Annotation

Annotate variants from a VCF file using a pre-built Ensembl VEP parquet cache.

In [1]:
import logging

logging.basicConfig(level=logging.INFO)

In [2]:
import vepyr

In [5]:
# Configure paths
# VCF_INPUT = "/Users/mwiewior/research/git/datafusion-bio-functions/vep-benchmark/data/HG002_chr1.vcf.gz"
VCF_INPUT = "/Users/mwiewior/research/git/datafusion-bio-functions/vep-benchmark/data/HG002_GRCh38_1_22_v4.2.1_benchmark.vcf.gz"
CACHE_DIR = "/Users/mwiewior/research/data/vep/wgs/fjall/115_GRCh38_vep"
REFERENCE_FASTA = (
    "/Users/mwiewior/research/data/vep/Homo_sapiens.GRCh38.dna.primary_assembly.fa"
)
WORK_DIR = "/Users/mwiewior/research/git/vepyr/sandbox"
VCF = "/Users/mwiewior/research/git/vepyr/sandbox/normalized.vcf.gz"

## Normalize, compress, and index the VCF

VEP requires **biallelic** variants. Use `bcftools norm -m -both` to decompose
multiallelic sites into separate records. Then bgzip-compress and tabix-index
the result for efficient random access.

In [6]:
import os
import subprocess

os.makedirs(WORK_DIR, exist_ok=True)

vcf_norm = os.path.join(WORK_DIR, "normalized.vcf")
vcf_gz = vcf_norm + ".gz"

# Step 1: Decompose multiallelic variants into biallelic records
print("Normalizing (bcftools norm -m -both) ...")
result = subprocess.run(
    ["bcftools", "norm", "-m", "-both", "-o", vcf_norm, VCF_INPUT],
    capture_output=True,
    text=True,
)
print(result.stderr.strip())
assert result.returncode == 0, f"bcftools norm failed: {result.stderr}"

# Step 2: Bgzip compress
print("Compressing (bgzip) ...")
subprocess.run(["bgzip", "-f", vcf_norm], check=True)
assert os.path.exists(vcf_gz)

# Step 3: Tabix index
print("Indexing (tabix) ...")
subprocess.run(["tabix", "-p", "vcf", vcf_gz], check=True)

# Count variants (use gunzip -c for cross-platform compatibility)
n_variants = int(
    subprocess.check_output(f"gunzip -c '{vcf_gz}' | grep -cv '^#'", shell=True).strip()
)
print(f"Ready: {n_variants:,} biallelic variants in {vcf_gz}")

Normalizing (bcftools norm -m -both) ...
Lines   total/split/joined/realigned/removed/skipped:	4048342/47781/0/0/0/0
Compressing (bgzip) ...
Indexing (tabix) ...
Ready: 4,096,123 biallelic variants in /Users/mwiewior/research/git/vepyr/sandbox/normalized.vcf.gz


In [8]:
vepyr.annotate(
    VCF_INPUT,
    CACHE_DIR,
    everything=True,
    reference_fasta=REFERENCE_FASTA,
    use_fjall=True,
    output_vcf="/tmp/annotated-vepyr.vcf",
)

INFO:vepyr:Running annotation on /Users/mwiewior/research/git/datafusion-bio-functions/vep-benchmark/data/HG002_GRCh38_1_22_v4.2.1_benchmark.vcf.gz with cache /Users/mwiewior/research/data/vep/wgs/fjall/115_GRCh38_vep


Annotating → annotated-vepyr.vcf: 0 variants [00:00, ? variants/s]

INFO:vepyr:Wrote 4048342 rows to /tmp/annotated-vepyr.vcf


'/tmp/annotated-vepyr.vcf'

In [4]:
lf = vepyr.annotate(
    VCF_INPUT,
    CACHE_DIR,
    everything=True,
    reference_fasta=REFERENCE_FASTA,
    use_fjall=True,
)

INFO:vepyr:Running annotation on /Users/mwiewior/research/git/datafusion-bio-functions/vep-benchmark/data/HG002_chr1.vcf.gz with cache /Users/mwiewior/research/data/vep/wgs/fjall/115_GRCh38_vep


In [6]:
lf.limit(5).collect()

chrom,start,end,id,ref,alt,qual,filter,most_severe_consequence,Allele,Consequence,IMPACT,SYMBOL,Gene,Feature_type,Feature,BIOTYPE,EXON,INTRON,HGVSc,HGVSp,cDNA_position,CDS_position,Protein_position,Amino_acids,Codons,Existing_variation,DISTANCE,STRAND,FLAGS,VARIANT_CLASS,SYMBOL_SOURCE,HGNC_ID,CANONICAL,MANE,MANE_SELECT,MANE_PLUS_CLINICAL,…,gnomADe_AMR_AF,gnomADe_ASJ_AF,gnomADe_EAS_AF,gnomADe_FIN_AF,gnomADe_MID_AF,gnomADe_NFE_AF,gnomADe_REMAINING_AF,gnomADe_SAS_AF,gnomADg_AF,gnomADg_AFR_AF,gnomADg_AMI_AF,gnomADg_AMR_AF,gnomADg_ASJ_AF,gnomADg_EAS_AF,gnomADg_FIN_AF,gnomADg_MID_AF,gnomADg_NFE_AF,gnomADg_REMAINING_AF,gnomADg_SAS_AF,MAX_AF,MAX_AF_POPS,CLIN_SIG,SOMATIC,PHENO,PUBMED,MOTIF_NAME,MOTIF_POS,HIGH_INF_POS,MOTIF_SCORE_CHANGE,TRANSCRIPTION_FACTORS,clin_sig_allele,clinical_impact,minor_allele,minor_allele_freq,clinvar_ids,cosmic_ids,dbsnp_ids
str,u32,u32,str,str,str,f64,str,str,str,list[str],str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,list[str],i64,i8,str,str,str,str,str,str,str,str,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,str,list[str],str,str,list[str],str,str,str,f32,list[str],list[str],str,str,f32,list[str],list[str],list[str]
"""chr1""",602113,602113,"""""","""T""","""TGCCCA""",50.0,"""PASS""","""intron_variant""","""GCCCA""","[""intron_variant"", ""non_coding_transcript_variant""]","""MODIFIER""",null,"""ENSG00000293331""","""Transcript""","""ENST00000634833""","""lncRNA""",null,"""2/5""","""ENST00000634833.2:n.176-537_17…",null,null,null,null,null,null,"[""rs528116099""]",null,-1,null,"""insertion""",null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""gnomADg_AFR&gnomADg_AMI&gnomAD…",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",604358,604358,"""""","""G""","""C""",50.0,"""PASS""","""intron_variant""","""C""","[""intron_variant"", ""non_coding_transcript_variant""]","""MODIFIER""",null,"""ENSG00000293331""","""Transcript""","""ENST00000634833""","""lncRNA""",null,"""2/5""","""ENST00000634833.2:n.176-2781C>…",null,null,null,null,null,null,"[""rs2808305""]",null,-1,null,"""SNV""",null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,0.003876,0.0,null,0.02128,0.0,0.0,0.0,null,0.001883,0.0,0.0,0.02128,"""gnomADg_AMR""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"[""rs2808305""]"
"""chr1""",604360,604360,"""""","""T""","""C""",50.0,"""PASS""","""intron_variant""","""C""","[""intron_variant"", ""non_coding_transcript_variant""]","""MODIFIER""",null,"""ENSG00000293331""","""Transcript""","""ENST00000634833""","""lncRNA""",null,"""2/5""","""ENST00000634833.2:n.176-2783A>…",null,null,null,null,null,null,"[""rs75816844""]",null,-1,null,"""SNV""",null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,0.0,0.0,null,0.0,0.0,0.0,0.0,null,0.0,0.0,0.0,0.0,"""gnomADg_AFR&gnomADg_AMR&gnomAD…",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"[""rs75816844""]"
"""chr1""",611317,611317,"""""","""A""","""G""",50.0,"""PASS""","""intron_variant""","""G""","[""intron_variant"", ""non_coding_transcript_variant""]","""MODIFIER""",null,"""ENSG00000293331""","""Transcript""","""ENST00000634833""","""lncRNA""",null,"""1/5""","""ENST00000634833.2:n.73+1424T>C""",null,null,null,null,null,null,"[""rs12025928""]",null,-1,null,"""SNV""",null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,0.9226,0.9717,0.8276,0.928,0.8682,0.9975,0.9318,0.8661,0.8987,0.9006,0.954,0.9975,"""gnomADg_EAS""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"[""rs12025928""]"
"""chr1""",631859,631860,"""""","""CG""","""C""",50.0,"""PASS""","""non_coding_transcript_exon_var…","""-""","[""non_coding_transcript_exon_variant""]","""MODIFIER""","""MTCO1P12""","""ENSG00000237973""","""Transcript""","""ENST00000414273""","""unprocessed_pseudogene""","""1/1""",null,"""ENST0000041

In [ ]:
df = lf.collect()
print(f"Variants: {df.height:,}")
print(f"Columns: {df.width}")
df.head(5)

## Inspect annotation columns

In [ ]:
# Key annotation columns
df.select(
    [
        "chrom",
        "start",
        "ref",
        "alt",
        "most_severe_consequence",
        "SYMBOL",
        "Gene",
        "IMPACT",
        "HGVSc",
        "HGVSp",
        "SIFT",
        "PolyPhen",
    ]
).head(10)

In [ ]:
# Allele frequencies
df.select(
    [
        "chrom",
        "start",
        "ref",
        "alt",
        "AF",
        "gnomADe_AF",
        "gnomADg_AF",
        "MAX_AF",
    ]
).head(10)

In [ ]:
# Consequence distribution
df.group_by("most_severe_consequence").len().sort("len", descending=True).head(15)

## Selective annotation

Instead of `everything=True`, you can pick individual flags:

In [ ]:
# HGVS + allele frequencies only
lf_selective = vepyr.annotate(
    VCF,
    CACHE_DIR,
    hgvs=True,
    af=True,
    af_gnomadg=True,
    reference_fasta=REFERENCE_FASTA,
)
lf_selective.collect().head(5)

## Filter and export

In [ ]:
import polars as pl

# Filter to high-impact variants
high_impact = (
    lf.filter(pl.col("IMPACT") == "HIGH")
    .select(
        [
            "chrom",
            "start",
            "ref",
            "alt",
            "SYMBOL",
            "most_severe_consequence",
            "HGVSc",
            "HGVSp",
        ]
    )
    .collect()
)
print(f"High-impact variants: {high_impact.height}")
high_impact.head(10)

In [ ]:
# Export to parquet
# df.write_parquet("/tmp/annotated.parquet")

# Export to CSV
# df.write_csv("/tmp/annotated.csv")

## Compare against Ensembl VEP golden standard

Load the Ensembl VEP 115 golden output and compare variant-level annotation
fields against our results.

In [ ]:
import polars_bio as pb
import os

# Bgzip + tabix the golden VCF if needed
GOLDEN_VCF = os.path.join(WORK_DIR, "HG002_chr1_0_vep115_golden.vcf")
GOLDEN_GZ = GOLDEN_VCF + ".gz"

if not os.path.exists(GOLDEN_GZ):
    subprocess.run(
        ["bgzip", "-c", GOLDEN_VCF], stdout=open(GOLDEN_GZ, "wb"), check=True
    )
    subprocess.run(["tabix", "-p", "vcf", GOLDEN_GZ], check=True)
    print(f"Prepared {GOLDEN_GZ}")

golden = pb.scan_vcf(GOLDEN_GZ).collect()
print(f"Golden: {golden.height:,} variants, {golden.width} columns")
print(f"Ours:   {df.height:,} variants, {df.width} columns")

In [ ]:
# Row count comparison
assert df.height == golden.height, (
    f"Row mismatch: ours={df.height}, golden={golden.height}"
)
print(f"Row count match: {df.height:,}")

In [ ]:
# Join on variant key (chrom, start, ref, alt) and compare shared columns
key = ["chrom", "start", "ref", "alt"]

# Find columns present in both (excluding VCF-specific like qual, filter, info)
ours_cols = set(df.columns) - set(key)
golden_cols = set(golden.columns) - set(key)
shared = sorted(ours_cols & golden_cols)
print(f"Shared columns ({len(shared)}): {shared}")

only_ours = sorted(ours_cols - golden_cols)
only_golden = sorted(golden_cols - ours_cols)
if only_ours:
    print(f"Only in ours ({len(only_ours)}): {only_ours}")
if only_golden:
    print(f"Only in golden ({len(only_golden)}): {only_golden}")

In [ ]:
# Per-column match rate for shared columns
# Join ours (suffixed _ours) with golden (suffixed _golden) on variant key
joined = df.select(key + [pl.col(c).alias(f"{c}_ours") for c in shared]).join(
    golden.select(key + [pl.col(c).alias(f"{c}_golden") for c in shared]),
    on=key,
    how="inner",
)
print(f"Joined: {joined.height:,} variants")

results = []
for col in shared:
    ours_c = f"{col}_ours"
    gold_c = f"{col}_golden"
    # Cast both to string for comparison (handles type mismatches)
    match_count = joined.filter(
        pl.col(ours_c).cast(pl.Utf8).fill_null("")
        == pl.col(gold_c).cast(pl.Utf8).fill_null("")
    ).height
    rate = match_count / joined.height if joined.height > 0 else 0
    results.append(
        {
            "column": col,
            "match_count": match_count,
            "total": joined.height,
            "match_rate": round(rate, 4),
        }
    )

comparison = pl.DataFrame(results).sort("match_rate")
print(f"\nPer-column match rates ({len(shared)} columns):")
comparison

In [ ]:
# Summary: columns with <100% match rate
mismatches = comparison.filter(pl.col("match_rate") < 1.0)
if mismatches.height == 0:
    print("All shared columns match perfectly!")
else:
    print(f"{mismatches.height} columns with mismatches:")
    mismatches